# 03 — Дополнительные эксперименты: AutoFE и последовательная модель

В этом ноутбуке проверяются дополнительные способы улучшить рекомендательную модель на данных Yambda. Основной ориентир — сильная модель LightGBM с расширенным пулом признаков `POOL2`, а основная метрика качества — **NDCG@10**.

## Что исследуется

**Часть 1 — анализ признаков**
- Качество на разных пользовательских сегментах и временных отрезках.
- Корреляционная и условная избыточность признаков.
- Графовые признаки и качество на новых пользовательско-трековых парах.
- Признаки явной обратной связи: лайки и дизлайки.
- Экспоненциальное затухание истории вместо фиксированных временных окон.
- Устойчивость отбора признаков с помощью RFE.

**Часть 2 — новые методы**
- Сравнение трёх стратегий автоматического поиска признаков: `random`, `evo` и `surrogate`.
- Объединение найденных признаков с `POOL2` и повторный отбор с помощью RFE.
- Построение последовательностей предыдущих взаимодействий пользователя.
- Обучение трансформера и проверка его предсказаний как дополнительного признака для LightGBM.

## Организация экспериментов

Параметр `PART` выбирает блок: `1` — анализ признаков, `2` — AutoFE и последовательная модель. `DEBUG = True` включает сокращённый запуск для проверки кода; полноценные эксперименты запускаются с `DEBUG = False`.

Ноутбук восстанавливает входные данные и кэши из предыдущих этапов. Результаты экспериментов сохраняются в каталоге `results/`.

> **Для интерпретации результатов:** в сохранённой конфигурации установлен `PART = 2`, поэтому блок `PART = 1` в этом запуске пропущен. Наличие его кода в ноутбуке не означает, что эти эксперименты были выполнены при данном запуске.


In [ ]:
import os, glob, json, time, re, gc, warnings, traceback, shutil
import numpy as np, pandas as pd, pyarrow.parquet as pq, lightgbm as lgb
warnings.filterwarnings('ignore', message='Overriding the parameters')
PART = 2  # 1 = ночь 1;  2 = ночь 2 (агенты на 10 seed + последовательная модель)
DEBUG = False  # True: облегчённый пробный прогон всего блокнота
SEED, N_USERS, POS_THRESHOLD = (42, 1500, 50)
DAY, SHIFT, TRAIN_DAYS = (86400, 26, 60)
SEARCH_ROOTS = ['/kaggle/input', '/kaggle/working', '.']

def find_file(name, required=True):
    for root in SEARCH_ROOTS:
        hits = glob.glob(os.path.join(root, '**', name), recursive=True)
        if hits:
            return hits[0]
    if required:
        raise FileNotFoundError(
            f'не найден файл {name}: подключите Output предыдущей версии через Add Input'
        )
    return None
RUN = dict(
    strata=PART == 1,
    redundancy=PART == 1,
    cond_redundancy=PART == 1,
    graph_compare=PART == 1,
    feedback=PART == 1,
    decay=PART == 1,
    stability=PART == 1,
    agents=PART == 2,
    seqmodel=PART == 2,
)
RUN_AGENTS = RUN['agents']
if DEBUG:
    ROUNDS, PER_ROUND, SEEDS_A, N_RES = (2, 3, [0], 2)
else:
    ROUNDS, PER_ROUND, SEEDS_A, N_RES = (10, 8, list(range(10)), 5)
os.makedirs('results', exist_ok=True)
print('PART =', PART, '| DEBUG =', DEBUG, '| этапы:', [k for k, v in RUN.items() if v])


## 1. Подготовка данных и базовых моделей


In [ ]:
BASE = find_file('base_listens.parquet')
df = pd.read_parquet(BASE)
print('df:', df.shape, '| колонки:', list(df.columns))
df['pr'] = (np.minimum(df['played_ratio_pct'], 100) / 100).astype(np.float32)
df['skip'] = (df['pr'] < 0.1).astype(np.float32)
df['full'] = (df['pr'] >= 0.9).astype(np.float32)
t_val, t_test = df['timestamp'].quantile([0.8, 0.9])
chk = np.where(df['timestamp'] < t_val, 'train', np.where(df['timestamp'] < t_test, 'val', 'test'))
assert (chk == df['split'].to_numpy()).all(), 'разбиение не совпадает с сохранённым'
FS = pd.read_parquet(find_file(f'features_snap_u{N_USERS}_s{SEED}.parquet'))
FW = pd.read_parquet(find_file(f'features_wide_u{N_USERS}_s{SEED}.parquet'))
FV2 = pd.read_parquet(find_file(f'features_v2_u{N_USERS}_s{SEED}.parquet'))
assert len(FS) == len(FW) == len(FV2) == len(df), 'кэши не совпадают с df по числу строк'
FEATS = list(FS.columns)
POOL2 = pd.concat([FS[[c for c in FEATS if c.startswith(('item_', 'gitem_'))]], FW, FV2], axis=1)
POOL2_COLS = list(POOL2.columns)
sets_json = json.load(open(find_file('feature_sets_v1.json')))
M2_COLS = list(sets_json['M2'])
assert (
    sorted(sets_json['M1']) == sorted(FEATS)
    and POOL2_COLS == sets_json['POOL2']
), 'наборы признаков не совпадают с кэшами'
sel_json = json.load(open(find_file('selected_sets_v1.json')))
sets_rfe = {int(k): v for k, v in sel_json['RFE'].items()}
sets_a1 = {int(k): v for k, v in sel_json['A1'].items()}
print(len(FEATS), len(M2_COLS), len(POOL2_COLS), '| RFE K:', sorted(sets_rfe))

# Подтягиваем исполнителей и альбомы для каждого трека.
def load_map(fname):
    p = find_file(fname, required=False)
    if p is None:
        from huggingface_hub import hf_hub_download
        p = hf_hub_download(
            repo_id='yandex/yambda',
            repo_type='dataset',
            filename=fname,
            local_dir='data',
        )
    m = pq.read_table(p).to_pandas()
    other = [c for c in m.columns if c != 'item_id']
    assert len(other) == 1
    return m.drop_duplicates('item_id').set_index('item_id')[other[0]]
for col, fname_ in [
    ('artist_id', 'artist_item_mapping.parquet'),
    ('album_id', 'album_item_mapping.parquet'),
]:
    s = df['item_id'].map(load_map(fname_)).astype('float64')
    df[col] = np.where(s.isna(), -1 - df['item_id'].astype(np.int64), s).astype(np.int64)
print('артистов/альбомов в df:', df['artist_id'].nunique(), df['album_id'].nunique())


In [ ]:
def add_window_feats2(
    df, keys, windows_days, name, target='label', snap=DAY,
    with_cnt=True, recency=False
):
    codes = df.groupby(keys, sort=False).ngroup().to_numpy(np.int64)
    ts = df['timestamp'].to_numpy(np.int64)
    val = df[target].to_numpy(np.float64)
    tq = ts if snap is None else ts // snap * snap
    comb = codes << SHIFT | ts
    order = np.argsort(comb, kind='stable')
    comb_s, codes_s, tq_s, val_s = (comb[order], codes[order], tq[order], val[order])
    cum = np.concatenate([[0.0], np.cumsum(val_s)])
    hi = np.searchsorted(comb_s, codes_s << SHIFT | tq_s, side='left')
    tag = 'rate' if target == 'label' else f'mean_{target}'
    out = {}
    for w in windows_days:
        lo = np.searchsorted(comb_s, codes_s << SHIFT | np.maximum(tq_s - w * DAY, 0), side='left')
        cnt = (hi - lo).astype(np.float64)
        mean = np.where(cnt > 0, (cum[hi] - cum[lo]) / np.maximum(cnt, 1), np.nan)
        if with_cnt:
            r = np.empty(len(df), np.float32)
            r[order] = cnt
            out[f'{name}_cnt_{w}d'] = r
        r = np.empty(len(df), np.float32)
        r[order] = mean
        out[f'{name}_{tag}_{w}d'] = r
    if recency:
        prev = hi - 1
        ok = prev >= 0
        prev = np.where(ok, prev, 0)
        ok &= codes_s[prev] == codes_s
        last_ts = comb_s[prev] & (1 << SHIFT) - 1
        rec = np.where(ok, (tq_s - last_ts) / DAY, np.nan)
        r = np.empty(len(df), np.float32)
        r[order] = rec
        out[f'{name}_recency_d'] = r
    return pd.DataFrame(out, index=df.index)

def ndcg_at_k(groups, scores, labels, k=10, min_size=10):
    d = pd.DataFrame({
        'g': np.asarray(groups),
        's': np.asarray(scores),
        'y': np.asarray(labels, dtype=float),
    })
    d = d[d.groupby('g')['y'].transform('size') >= min_size]
    a = d.sort_values(['g', 's'], ascending=[True, False])
    ra = a.groupby('g').cumcount().to_numpy() + 1
    dcg = pd.Series(
        a['y'].to_numpy() / np.log2(ra + 1) * (ra <= k), index=a.index
    ).groupby(a['g']).sum()
    b = d.sort_values(['g', 'y'], ascending=[True, False])
    rb = b.groupby('g').cumcount().to_numpy() + 1
    idcg = pd.Series(
        b['y'].to_numpy() / np.log2(rb + 1) * (rb <= k), index=b.index
    ).groupby(b['g']).sum()
    return (dcg / idcg).dropna().mean()

def ndcg_groups(groups, scores, labels, k=10, min_size=10):
    d = pd.DataFrame({'g': np.asarray(groups),
         's': np.asarray(scores),
        'y': np.asarray(labels, dtype=float)})
    d = d[d.groupby('g')['y'].transform('size') >= min_size]
    a = d.sort_values(['g', 's'], ascending=[True, False])
    ra = a.groupby('g').cumcount().to_numpy() + 1
    dcg = pd.Series(a['y'].to_numpy() / np.log2(ra + 1) * (ra <= k),
        index=a.index).groupby(a['g']).sum()
    b = d.sort_values(['g', 'y'], ascending=[True, False])
    rb = b.groupby('g').cumcount().to_numpy() + 1
    idcg = pd.Series(b['y'].to_numpy() / np.log2(rb + 1) * (rb <= k),
        index=b.index).groupby(b['g']).sum()
    return (dcg / idcg).dropna()

def prep(mask):
    d = df.loc[mask, ['uid', 'timestamp', 'label']].copy()
    d['g'] = d['uid'].astype(np.int64) * 1000 + d['timestamp'].astype(np.int64) // DAY
    d = d.sort_values('g', kind='stable')
    return (d, d.groupby('g', sort=True).size().to_numpy())
params_r = dict(
    objective='lambdarank',
    metric='ndcg',
    eval_at=[10],
    learning_rate=0.05,
    num_leaves=63,
    min_data_in_leaf=200,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=1,
    verbose=-1,
)
# Настройки LightGBM для длительного отбора признаков.
LGB_THREADS = 4
params_r = dict(
    objective='lambdarank',
    metric='ndcg',
    eval_at=[10],
    learning_rate=0.05,
    num_leaves=63,
    min_data_in_leaf=200,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=1,
    verbosity=-1,
    num_threads=LGB_THREADS,
)

def fit_sel(cols, seed=0, rounds=300, max_bin=63, tag='fit'):
    """
    Обучение LambdaRank и оценка на sel/rep.

    Важные изменения:
    - ограничиваем число CPU-потоков;
    - печатаем начало/конец обучения;
    - явно освобождаем LightGBM Dataset;
    - модель возвращается наружу только если она действительно нужна.
    """
    cols = list(cols)
    ci = [CP[c] for c in cols]
    print(f'[LightGBM START] {tag} | features={len(cols)} | rounds={rounds}', flush=True)
    dtr = lgb.Dataset(
        Xtr_p[:, ci],
        label=d_tr['label'].to_numpy(),
        group=sz_tr,
        params={'max_bin': max_bin},
        free_raw_data=True,
    )
    dsel = lgb.Dataset(
        Xsel_p[:, ci],
        label=d_sel['label'].to_numpy(),
        group=sz_sel,
        reference=dtr,
        free_raw_data=True,
    )
    model_params = {**params_r, 'seed': seed, 'num_threads': LGB_THREADS}
    try:
        m = lgb.train(
            model_params,
            dtr,
            num_boost_round=rounds,
            valid_sets=[dsel],
            callbacks=[
                lgb.early_stopping(30, verbose=False),
                lgb.log_evaluation(0),
            ],
        )
        best_iter = m.best_iteration or rounds
        p_sel = m.predict(Xsel_p[:, ci], num_iteration=best_iter)
        p_rep = m.predict(Xrep_p[:, ci], num_iteration=best_iter)
        s_sel = ndcg_at_k(d_sel['g'], p_sel, d_sel['label'])
        s_rep = ndcg_at_k(d_rep['g'], p_rep, d_rep['label'])
        print(
            f'[LightGBM DONE] {tag} | best_iter={best_iter} | '
            f'sel={s_sel:.4f} | rep={s_rep:.4f}',
            flush=True,
        )
        return (m, s_sel, s_rep, p_rep)
    finally:
        # Освобождаем датасеты даже при ошибке.
        try:
            del dtr
        except Exception:
            pass
        try:
            del dsel
        except Exception:
            pass
        gc.collect()

def boot_diff_rep(pred_a, pred_b, n_boot=1000, seed=0):
    ga = ndcg_groups(d_rep['g'], pred_a, d_rep['label'])
    gb = ndcg_groups(d_rep['g'], pred_b, d_rep['label'])
    diff = (gb - ga).dropna()
    pu = diff.groupby(diff.index // 1000).agg(['sum', 'count'])
    sums, cnts = (pu['sum'].to_numpy(), pu['count'].to_numpy())
    r = np.random.default_rng(seed)
    est = []
    for _ in range(n_boot):
        idx = r.integers(0, len(sums), len(sums))
        est.append(sums[idx].sum() / cnts[idx].sum())
    return (diff.mean(), np.percentile(est, [2.5, 97.5]))

def rfe_path(start_cols, targets, drop_frac=0.15, rounds=220):
    """
    Безопасный RFE.

    Для RFE используем немного меньше boosting rounds,
    потому что здесь модель обучается многократно.

    Метод отбора остаётся тем же:
    на каждом шаге удаляем наименее важные признаки
    по gain.
    """
    cols = list(dict.fromkeys(start_cols))
    out = {}
    targets = sorted(set(targets), reverse=True)
    for K in targets:
        step = 0
        while len(cols) > K:
            step += 1
            old_n = len(cols)
            print(f'[RFE] target K={K} | step={step} | {old_n} -> ...', flush=True)
            m, _, _, _ = fit_sel(cols, seed=0, rounds=rounds, tag=f'RFE K={K} step={step}')
            imp = pd.Series(m.feature_importance('gain'), index=cols).sort_values(ascending=False)
            new_n = max(K, int(len(cols) * (1 - drop_frac)))
            cols = imp.index[:new_n].tolist()
            print(f'[RFE] K={K}: {old_n} -> {len(cols)}', flush=True)
            # Освобождаем модель перед следующим шагом.
            del m
            del imp
            gc.collect()
        out[K] = list(cols)
        print(f'[RFE] ГОТОВ K={K}: {len(out[K])} признаков', flush=True)
    return out


In [ ]:
tr = (df['split'] == 'train') & (df['timestamp'] >= t_val - TRAIN_DAYS * DAY)
va = df['split'] == 'val'
d_tr, sz_tr = prep(tr)
rs = np.random.default_rng(123)
val_users = df.loc[va, 'uid'].unique()
half = set(rs.choice(val_users, size=len(val_users) // 2, replace=False).tolist())
is_sel = df['uid'].isin(half).to_numpy()
d_sel, sz_sel = prep(va & is_sel)
d_rep, sz_rep = prep(va & ~is_sel)
print(
    'групп sel/rep:', len(sz_sel), len(sz_rep),
    '| пересечение пользователей:', len(set(d_sel.uid) & set(d_rep.uid)),
)
idx_tr_all, idx_sel, idx_rep = (d_tr.index.to_numpy(),
     d_sel.index.to_numpy(),
    d_rep.index.to_numpy())
XP = POOL2.to_numpy(dtype=np.float32)
CP = {c: i for i, c in enumerate(POOL2_COLS)}
Xtr_p, Xsel_p, Xrep_p = (XP[idx_tr_all], XP[idx_sel], XP[idx_rep])
rnd = ndcg_at_k(d_rep['g'], np.random.default_rng(0).random(len(d_rep)), d_rep['label'])
v = XP[idx_rep, CP['gitem_rate_30d']]
pop = ndcg_at_k(d_rep['g'], np.where(np.isnan(v), 0.64, v), d_rep['label'])
print(f'rep: случайный {rnd:.4f} | популярность {pop:.4f}')
del XP, POOL2, FS, FW, FV2
gc.collect()  # освобождаем память: всё нужное уже в Xtr_p/Xsel_p/Xrep_p
R = {}
for name, cols in [('M1', FEATS), ('M2', M2_COLS), ('POOL2', POOL2_COLS)]:
    m, s_sel, s_rep, p = fit_sel(list(cols))
    R[name] = dict(m=m, sel=s_sel, rep=s_rep, p=p, n=len(cols))
    print(f'{name:6s} ({len(cols)}): NDCG@10 sel={s_sel:.4f} rep={s_rep:.4f}', flush=True)


## 2. Автоматический поиск признаков: язык кандидатов и агенты


In [ ]:
from sklearn.ensemble import RandomForestRegressor
KEYS = {
    'user': ['uid'],
    'item': ['item_id'],
    'ui': ['uid', 'item_id'],
    'uart': ['uid', 'artist_id'],
    'ualb': ['uid', 'album_id'],
    'art': ['artist_id'],
    'alb': ['album_id'],
}
AGGS = ['cnt', 'rate', 'mean_pr', 'skip_rate', 'full_rate', 'recency']
TARGET_OF = {'rate': 'label', 'mean_pr': 'pr', 'skip_rate': 'skip', 'full_rate': 'full'}
NEW = {}  # имя признака -> значения для всех строк df
KEEP = set()  # признаки, принятые хотя бы одним запуском (их массивы нельзя удалять)

def validate(p):
    try:
        key, agg = (str(p['key']), str(p['agg']))
        w = int(p.get('window', 1)) if agg != 'recency' else 0
    except Exception:
        return None
    if key not in KEYS or agg not in AGGS:
        return None
    return {'key': key, 'agg': agg, 'window': min(max(w, 1), 400) if agg != 'recency' else 0}

def fname(p):
    return (
        f"{p['key']}_recency_d"
        if p['agg'] == 'recency'
        else f"{p['key']}_{p['agg']}_{p['window']}d"
    )
_name_re = re.compile(
    '^(user|item|ui|uart|ualb|art|alb)_(cnt|rate|mean_pr|skip_rate|full_rate)_(\\d+)d$'
)
_rec_re = re.compile('^(user|item|ui|uart|ualb|art|alb)_recency_d$')

def parse_name(n):
    m = _name_re.match(n)
    if m:
        return validate({'key': m.group(1), 'agg': m.group(2), 'window': int(m.group(3))})
    m = _rec_re.match(n)
    if m:
        return validate({'key': m.group(1), 'agg': 'recency', 'window': 0})
    return None

def compile_programs(progs):
    groups = {}
    for p in progs:
        if fname(p) in NEW:
            continue
        kind = (
            'recency' if p['agg'] == 'recency'
            else 'cnt' if p['agg'] == 'cnt'
            else TARGET_OF[p['agg']]
        )
        groups.setdefault((p['key'], kind), []).append(p)
    for (key, kind), ps in groups.items():
        keys = KEYS[key]
        if kind == 'recency':
            F = add_window_feats2(df, keys, [], key, recency=True)
            NEW[f'{key}_recency_d'] = F[f'{key}_recency_d'].to_numpy()
        elif kind == 'cnt':
            ws = sorted({p['window'] for p in ps})
            F = add_window_feats2(df, keys, ws, key, target='label', with_cnt=True)
            for w in ws:
                NEW[f'{key}_cnt_{w}d'] = F[f'{key}_cnt_{w}d'].to_numpy()
        else:
            ws = sorted({p['window'] for p in ps})
            tag = 'rate' if kind == 'label' else f'mean_{kind}'
            F = add_window_feats2(df, keys, ws, key, target=kind, with_cnt=False)
            for p in ps:
                NEW[fname(p)] = F[f"{key}_{tag}_{p['window']}d"].to_numpy()

def ensure_features(names):
    progs = [parse_name(n) for n in names if n not in NEW]
    compile_programs([p for p in progs if p is not None])
    KEEP.update(names)
# Прокси-модель быстро оценивает кандидатов.
PROXY_FRAC, PROXY_TREES = (0.2, 60)
ru = np.random.default_rng(7)
tr_users = d_tr['uid'].unique()
u_sub = set(ru.choice(tr_users, size=int(len(tr_users) * PROXY_FRAC), replace=False).tolist())
m_sub = d_tr['uid'].isin(u_sub).to_numpy()
y_sub = d_tr['label'].to_numpy()[m_sub]
sz_sub = d_tr.loc[m_sub].groupby('g', sort=True).size().to_numpy()
params_px = {**params_r, 'learning_rate': 0.15, 'num_leaves': 31, 'seed': 0}
idx_tr_sub = idx_tr_all[m_sub]
ci_base = [CP[c] for c in FEATS]

def proxy_score(Xtr, Xsel, seed=0):
    dtr = lgb.Dataset(Xtr, y_sub, group=sz_sub, params={'max_bin': 31})
    m = lgb.train({**params_px, 'seed': seed}, dtr, PROXY_TREES)
    return ndcg_at_k(d_sel['g'], m.predict(Xsel), d_sel['label'])
Xb_tr0, Xb_sel0 = (Xtr_p[m_sub][:, ci_base], Xsel_p[:, ci_base])
noise = [proxy_score(Xb_tr0, Xb_sel0, seed=s) for s in (0, 1, 2)]
THR = max(0.0005, 2 * float(np.std(noise)))
print(f'прокси на M1: {np.mean(noise):.4f} ± {np.std(noise):.4f}; порог принятия Δ > {THR:.4f}')

# Стратегии генерируют кандидатов для проверки.
def sample_program(rng):
    return validate({
        'key': str(rng.choice(list(KEYS))),
        'agg': str(rng.choice(AGGS)),
        'window': int(np.exp(rng.uniform(0, np.log(400)))),
    })

def propose_random(history, n, seen, rng):
    out = []
    while len(out) < n:
        p = sample_program(rng)
        if fname(p) not in seen and all((fname(p) != fname(q) for q in out)):
            out.append(p)
    return out

def propose_evo(history, n, seen, rng):
    good = [h['p'] for h in history if h['delta'] > 0]
    out, tries = ([], 0)
    while len(out) < n and tries < 200:
        tries += 1
        if good and rng.random() < 0.6:
            p = dict(good[int(rng.integers(len(good)))])
            r = rng.random()
            if r < 0.5 and p['agg'] != 'recency':
                p['window'] = int(round(p['window'] * float(rng.choice([0.5, 0.7, 1.4, 2.0]))))
            elif r < 0.75:
                p['key'] = str(rng.choice(list(KEYS)))
            else:
                p['agg'] = str(rng.choice(AGGS))
            p = validate(p)
        else:
            p = sample_program(rng)
        if p and fname(p) not in seen and all((fname(p) != fname(q) for q in out)):
            out.append(p)
    return out

def encode(p):
    v = [0.0] * (len(KEYS) + len(AGGS) + 1)
    v[list(KEYS).index(p['key'])] = 1.0
    v[len(KEYS) + AGGS.index(p['agg'])] = 1.0
    v[-1] = float(np.log1p(p['window']))
    return v

def propose_surrogate(history, n, seen, rng, kappa=1.0, n_cand=300, warmup=16):
    if len(history) < warmup:
        return propose_random(history, n, seen, rng)
    X = np.array([encode(h['p']) for h in history])
    y = np.array([h['delta'] for h in history])
    rf = RandomForestRegressor(
        n_estimators=100,
        min_samples_leaf=2,
        random_state=int(rng.integers(1000000)),
    ).fit(X, y)
    cands = propose_random(history, n_cand, seen, rng)
    Xc = np.array([encode(p) for p in cands])
    preds = np.stack([t.predict(Xc) for t in rf.estimators_])
    score = preds.mean(0) + kappa * preds.std(0)
    return [cands[i] for i in np.argsort(-score)[:n]]

def run_agent(propose, rounds=10, per_round=8, seed=0):
    rng = np.random.default_rng(seed)
    Xtr_b, Xsel_b = (Xb_tr0.copy(), Xb_sel0.copy())
    base = float(np.mean(noise))
    seen, accepted, history, t0 = (set(FEATS), [], [], time.time())
    for r in range(rounds):
        progs = propose(history, per_round, seen, rng)
        if not progs:
            print(f'  раунд {r + 1}: нет предложений')
            continue
        compile_programs(progs)
        best = None
        for p in progs:
            nm = fname(p)
            seen.add(nm)
            s = proxy_score(
                np.column_stack([Xtr_b, NEW[nm][idx_tr_sub]]),
                np.column_stack([Xsel_b, NEW[nm][idx_sel]]),
            )
            history.append({'p': p, 'delta': float(s - base)})
            if best is None or s - base > best[1]:
                best = (p, s - base, s)
        if best[1] > THR:
            p, d_, s = best
            Xtr_b = np.column_stack([Xtr_b, NEW[fname(p)][idx_tr_sub]])
            Xsel_b = np.column_stack([Xsel_b, NEW[fname(p)][idx_sel]])
            accepted.append(fname(p))
            KEEP.add(fname(p))
            base = s
        for p in progs:
            if fname(p) not in KEEP:
                NEW.pop(fname(p), None)
        print(
            f"  раунд {r + 1}: лучший {fname(best[0])} Δ={best[1]:+.4f}"
            f"{' (принят)' if best[1] > THR else ''}; принято {len(accepted)}",
            flush=True,
        )
    return dict(accepted=accepted, history=history, evals=len(history), sec=time.time() - t0)

def run_agent_cached(name, prop, seed):
    path_ = f'results/agent_{name}_s{seed}.json'
    if os.path.exists(path_):
        res = json.load(open(path_))
        ensure_features(res['accepted'])
        return res
    res = run_agent(prop, ROUNDS, PER_ROUND, seed=seed)
    json.dump(res, open(path_, 'w'), ensure_ascii=False)
    return res

def final_eval(names, seed=0):
    ensure_features(names)

    def X(Xp, idx):
        return np.column_stack([Xp[:, ci_base]] + [NEW[n][idx] for n in names])
    Xtr, Xs, Xr = (X(Xtr_p, idx_tr_all), X(Xsel_p, idx_sel), X(Xrep_p, idx_rep))
    dtr = lgb.Dataset(Xtr, d_tr['label'].to_numpy(), group=sz_tr, params={'max_bin': 63})
    dsel = lgb.Dataset(Xs, d_sel['label'].to_numpy(), group=sz_sel, reference=dtr)
    m = lgb.train(
        {**params_r, 'seed': seed},
        dtr,
        300,
        valid_sets=[dsel],
        callbacks=[
            lgb.early_stopping(30, verbose=False),
            lgb.log_evaluation(0),
        ],
    )
    p_rep = m.predict(Xr, num_iteration=m.best_iteration)
    return (ndcg_at_k(d_rep['g'], p_rep, d_rep['label']), p_rep)


## 3. Графовые признаки: вложения треков и векторы вкуса


In [ ]:
import scipy.sparse as sp
from scipy.sparse.linalg import svds

def spectral_item_vectors(u, i, d=16, min_deg=5):
    ucodes, uidx = pd.factorize(u)
    icodes, iidx = pd.factorize(i)
    M = sp.coo_matrix(
        (np.ones(len(ucodes), np.float32), (ucodes, icodes)),
        shape=(len(uidx), len(iidx)),
    ).tocsr()
    M.data = np.minimum(M.data, 1.0)
    deg_i = np.asarray(M.sum(0)).ravel()
    keep_ = deg_i >= min_deg
    M, iidx = (M[:, keep_], iidx[keep_])
    du = np.asarray(M.sum(1)).ravel()
    di = np.asarray(M.sum(0)).ravel()
    M = sp.diags(1 / np.sqrt(np.maximum(du, 1))) @ M @ sp.diags(1 / np.sqrt(np.maximum(di, 1)))
    _, S, Vt = svds(M.astype(np.float64), k=d)
    V = Vt.T * S
    V = V / (np.linalg.norm(V, axis=1, keepdims=True) + 1e-09)
    return (pd.Series(np.arange(len(iidx)), index=pd.Index(iidx)), V.astype(np.float32))

def taste_feats(df, vec_index, V, windows_days, name, snap=DAY, chunk=1000000):
    n, d = (len(df), V.shape[1])
    pos = vec_index.reindex(df['item_id'].to_numpy()).to_numpy(dtype=np.float64)
    has = ~np.isnan(pos)
    E = V[np.where(has, pos, 0).astype(np.int64)].astype(np.float32)
    E[~has] = 0.0
    codes = df.groupby('uid', sort=False).ngroup().to_numpy(np.int64)
    ts = df['timestamp'].to_numpy(np.int64)
    tq = ts // snap * snap
    comb = codes << SHIFT | ts
    order = np.argsort(comb, kind='stable')
    comb_s, codes_s, tq_s = (comb[order], codes[order], tq[order])
    E_s, has_s = (E[order], has[order])
    lab_s = df['label'].to_numpy()[order].astype(np.float64)
    hi = np.searchsorted(comb_s, codes_s << SHIFT | tq_s, side='left')
    los = {
        w: np.searchsorted(
            comb_s, codes_s << SHIFT | np.maximum(tq_s - w * DAY, 0), side='left'
        )
        for w in windows_days
    }
    out = {}
    for sign, wt in (('pos', lab_s), ('neg', 1.0 - lab_s)):
        wt = wt * has_s
        cumw = np.concatenate([[0.0], np.cumsum(wt)])
        cumv = np.zeros((n + 1, d), np.float64)
        np.cumsum(E_s * wt[:, None], axis=0, out=cumv[1:])
        for w in windows_days:
            res_s = np.full(n, np.nan, np.float32)
            lo = los[w]
            for a in range(0, n, chunk):
                b = min(n, a + chunk)
                S = cumv[hi[a:b]] - cumv[lo[a:b]]
                W = cumw[hi[a:b]] - cumw[lo[a:b]]
                nrm = np.linalg.norm(S, axis=1)
                dot = np.einsum('ij,ij->i', S, E_s[a:b].astype(np.float64))
                ok = (W > 0) & (nrm > 1e-09) & has_s[a:b]
                res_s[a:b] = np.where(ok, dot / np.maximum(nrm, 1e-09), np.nan)
            r = np.empty(n, np.float32)
            r[order] = res_s
            out[f'{name}_taste_{sign}_{w}d'] = r
        del cumv
    res = pd.DataFrame(out, index=df.index)
    for w in windows_days:
        res[f'{name}_taste_diff_{w}d'] = (
            res[f'{name}_taste_pos_{w}d'] - res[f'{name}_taste_neg_{w}d']
        ).astype(np.float32)
    return res
GRAPH_PATH, VEC_PATH = ('features_graph_v1.parquet', 'graph_item_vectors_v1.npz')
T0 = int(t_val - TRAIN_DAYS * DAY)
vec_file = find_file(VEC_PATH, required=False)
if vec_file:
    z = np.load(vec_file)
    vec_index = pd.Series(np.arange(len(z['ids'])), index=pd.Index(z['ids']))
    V_graph = z['V']
    print('вложения треков загружены:', V_graph.shape)
else:
    RAW = find_file('multi_event.parquet', required=False)
    if RAW is None:
        from huggingface_hub import hf_hub_download
        RAW = hf_hub_download(
            repo_id='yandex/yambda',
            repo_type='dataset',
            filename='flat/50m/multi_event.parquet',
            local_dir='data',
        )
    tab = pq.read_table(
        RAW,
        columns=['uid', 'item_id', 'timestamp', 'played_ratio_pct'],
        filters=[
            ('event_type', '==', 'listen'),
            ('timestamp', '<', T0),
            ('played_ratio_pct', '>=', POS_THRESHOLD),
        ],
    )
    past = tab.to_pandas()
    del tab
    print(len(past), 'дослушанных прослушиваний до T0')
    vec_index, V_graph = spectral_item_vectors(
        past['uid'].to_numpy(), past['item_id'].to_numpy(), d=16, min_deg=5
    )
    del past
    np.savez(VEC_PATH, ids=vec_index.index.to_numpy(), V=V_graph)
    print(
        'вложений треков:', V_graph.shape,
        '| покрытие строк df:', round(df['item_id'].isin(vec_index.index).mean(), 3),
    )
cached = find_file(GRAPH_PATH, required=False)
if cached:
    TASTE = pd.read_parquet(cached)
    print('граф-признаки загружены из кэша:', cached)
else:
    TASTE = taste_feats(df, vec_index, V_graph, [30, 90, 400], 'graph')
    TASTE.to_parquet(GRAPH_PATH)
assert len(TASTE) == len(df)
print(TASTE.notna().mean().round(3).to_dict())


## 4. Общие функции для сравнения моделей


In [ ]:
def register_dict(feat_dict):
    """Добавляет признаки (массивы длины len(df)) в матрицы train/sel/rep; хранит массивы в NEW."""
    global Xtr_p, Xsel_p, Xrep_p
    names = [n for n in feat_dict if n not in CP]
    for k, n in enumerate(names):
        CP[n] = Xtr_p.shape[1] + k
        NEW[n] = feat_dict[n]
        KEEP.add(n)
    if names:
        Xtr_p = np.column_stack([Xtr_p] + [feat_dict[n][idx_tr_all] for n in names])
        Xsel_p = np.column_stack([Xsel_p] + [feat_dict[n][idx_sel] for n in names])
        Xrep_p = np.column_stack([Xrep_p] + [feat_dict[n][idx_rep] for n in names])
    return names

def get_pred(key, cols):
    if key not in R or 'p' not in R[key]:
        _, _, s_rep, p = fit_sel(list(cols))
        R[key] = dict(p=p, rep=s_rep)
    return R[key]['p']
g_rep, y_rep = (d_rep['g'].to_numpy(), d_rep['label'].to_numpy())

def strat_diff(pa, pb, mask, n_boot=500, seed=0, min_groups=30):
    ga = ndcg_groups(g_rep[mask], pa[mask], y_rep[mask])
    gb = ndcg_groups(g_rep[mask], pb[mask], y_rep[mask])
    diff = (gb - ga).dropna()
    if len(diff) < min_groups:
        return None
    pu = diff.groupby(diff.index // 1000).agg(['sum', 'count'])
    s, c = (pu['sum'].to_numpy(), pu['count'].to_numpy())
    r = np.random.default_rng(seed)
    est = []
    for _ in range(n_boot):
        i = r.integers(0, len(s), len(s))
        est.append(s[i].sum() / c[i].sum())
    lo, hi = np.percentile(est, [2.5, 97.5])
    return dict(групп=len(diff), A=ga.mean(), B=gb.mean(), d=diff.mean(), lo=lo, hi=hi)

def compare_rfe_generic(pool_cols, tag, base_sets, Ks=(48, 26), rfe_rounds=220):
    """
    Безопасное сравнение расширенного пула с базовыми наборами.

    Особенности:
    1. RFE выполняется последовательно.
    2. Базовая модель для K не переобучается,
       если её предсказания уже есть в R.
    3. После каждого K сохраняется промежуточный CSV.
    4. После каждого обучения освобождается память.
    5. Подробный прогресс выводится в Kaggle.
    """
    t0 = time.time()
    pool_cols = list(dict.fromkeys(pool_cols))
    Ks = tuple(sorted(Ks, reverse=True))
    print(
        '\n' + '=' * 70
        + f"\nRFE: '{tag}'"
        + f'\nПризнаков в расширенном пуле: {len(pool_cols)}'
        + f'\nЦелевые K: {Ks}'
        + f'\nRFE rounds: {rfe_rounds}'
        + '\n' + '=' * 70,
        flush=True,
    )
    # Сначала отбираем признаки из расширенного пула.
    print('\n[1/4] Запуск RFE на расширенном пуле...', flush=True)
    new_sets = rfe_path(pool_cols, list(Ks), drop_frac=0.15, rounds=rfe_rounds)
    print('[1/4] RFE завершён за', round(time.time() - t0), 'с', flush=True)
    # Затем сравниваем новые наборы с базовыми.
    rows = []
    for K in Ks:
        print(
            '\n' + '-' * 60
            + f'\n[2/4] K={K}'
            + f'\nБаза: {len(base_sets[K])} признаков'
            + f'\nНовый набор: {len(new_sets[K])} признаков',
            flush=True,
        )
        # Базовая модель.
        base_key = f'база_{K}_{tag}'
        if base_key in R and 'p' in R[base_key]:
            print(f'[CACHE] Используем уже обученную базу: {base_key}', flush=True)
            base_rep = R[base_key]['rep']
            p_base = R[base_key]['p']
        else:
            # Повторно используем готовые предсказания,
            # если они уже сохранены в памяти.
            generic_key = None
            for candidate in [f'RFE_{K}', f'rfe_{K}', f'база_{K}']:
                if candidate in R and 'p' in R[candidate]:
                    generic_key = candidate
                    break
            if generic_key is not None:
                print(f'[CACHE] Используем {generic_key}', flush=True)
                base_rep = R[generic_key]['rep']
                p_base = R[generic_key]['p']
            else:
                print(f'[LightGBM] Обучаем базу K={K}', flush=True)
                m_base, s_sel_base, base_rep, p_base = fit_sel(
                    list(base_sets[K]), seed=0, rounds=300, tag=f'BASE K={K}'
                )
                # Освобождаем модель после оценки.
                del m_base
                gc.collect()
            R[base_key] = dict(p=p_base, rep=base_rep)
        print(f'[BASE] K={K}: rep={base_rep:.4f}', flush=True)
        # Модель с новым набором признаков.
        new_key = f'{tag}_{K}_{tag}'
        print(f'[LightGBM] Обучаем новый набор K={K}', flush=True)
        m_new, s_sel_new, new_rep, p_new = fit_sel(
            list(new_sets[K]), seed=0, rounds=300, tag=f'{tag} K={K}'
        )
        R[new_key] = dict(p=p_new, rep=new_rep)
        print(f'[NEW] K={K}: sel={s_sel_new:.4f} | rep={new_rep:.4f}', flush=True)
        # Сравниваем предсказания и доверительный интервал.
        mean, ci = boot_diff_rep(p_base, p_new)
        print(f'[DIFF] K={K}: {tag} - база = {mean:+.4f} [{ci[0]:+.4f}; {ci[1]:+.4f}]', flush=True)
        rows.append(dict(K=K, d=mean, lo=ci[0], hi=ci[1], база=base_rep, новый=new_rep))
        # Сохраняем прогресс после каждого размера набора.
        partial = pd.DataFrame(rows)
        partial.to_csv(f'results/compare_rfe_{tag}_partial.csv', index=False)
        # Модель больше не нужна.
        del m_new
        del p_new
        gc.collect()
        print(f'[SAVE] промежуточный результат K={K} сохранён', flush=True)
    # Записываем итоговые наборы и метрики.
    result_df = pd.DataFrame(rows)
    result_df.to_csv(f'results/compare_rfe_{tag}.csv', index=False)
    with open(f'results/selected_sets_{tag}.json', 'w', encoding='utf-8') as f:
        json.dump({str(k): v for k, v in new_sets.items()}, f, ensure_ascii=False, indent=2)
    # Выводим итоговую таблицу.
    print(
        '\n' + '=' * 70
        + f"\nRFE '{tag}' ЗАВЕРШЁН"
        + f'\nВремя: {round(time.time() - t0)} с'
        + '\n' + '=' * 70,
        flush=True,
    )
    print(result_df.to_string(index=False), flush=True)
    gc.collect()
    return new_sets
print(
    'базовые модели готовы; rep: M1 %.4f, M2 %.4f, пул v2 %.4f'
    % (R['M1']['rep'], R['M2']['rep'], R['POOL2']['rep'])
)


## 5. E1 — Анализ сегментов и временных отрезков


In [ ]:
if RUN['strata']:
    print('=== E1 страты и недели ===', flush=True)
    _t0 = time.time()
    try:
        P_M1, P_M2 = (get_pred('M1', FEATS), get_pred('M2', M2_COLS))
        P_POOL = get_pred('POOL2', POOL2_COLS)
        P_R26 = get_pred('RFE26_v2', sets_rfe[26])
        mean, ci = boot_diff_rep(P_M2, P_R26)
        print(f'RFE-26 - M2 (разность на rep): {mean:+.4f} [{ci[0]:+.4f}; {ci[1]:+.4f}]')
        pd.DataFrame(
            [dict(сравнение='RFE-26 - M2', d=mean, lo=ci[0], hi=ci[1])]
        ).to_csv('results/rfe26_vs_m2.csv', index=False)
        col = lambda name: Xrep_p[:, CP[name]]

        def terciles(x):
            q1, q2 = np.nanquantile(x, [1 / 3, 2 / 3])
            return [('низкая', x <= q1), ('средняя', (x > q1) & (x <= q2)), ('высокая', x > q2)]
        strata = [
            ('пары: новые (ui_cnt_400d=0)', col('ui_cnt_400d') == 0),
            ('пары: повторные', col('ui_cnt_400d') > 0),
        ]
        strata += [
            (f'популярность трека (gitem_cnt_90d): {n}', m)
            for n, m in terciles(col('gitem_cnt_90d'))
        ]
        strata += [
            (f'активность пользователя (user_cnt_30d): {n}', m)
            for n, m in terciles(col('user_cnt_30d'))
        ]
        rows = []
        for cmp_name, pa, pb in [('RFE-26 - M1', P_M1, P_R26), ('пул v2 - M2', P_M2, P_POOL)]:
            for sname, mk in strata:
                r = strat_diff(pa, pb, mk)
                if r:
                    rows.append(
                        dict(
                            сравнение=cmp_name,
                            страта=sname,
                            доля_строк=round(float(mk.mean()), 3),
                            **r,
                        )
                    )
        strat_df = pd.DataFrame(rows)
        print(strat_df.round(4).to_string(index=False))
        strat_df.to_csv('results/strata_diff.csv', index=False)
        wk = d_rep['timestamp'].to_numpy().astype(np.int64) // (7 * DAY)
        rows = []
        for w in sorted(set(wk)):
            mk = wk == w
            for cmp_name, pa, pb in [('RFE-26 - M1', P_M1, P_R26), ('пул v2 - M2', P_M2, P_POOL)]:
                r = strat_diff(pa, pb, mk)
                if r:
                    rows.append(
                        dict(
                            неделя=int(w), сравнение=cmp_name,
                            доля_строк=round(float(mk.mean()), 3), **r,
                        )
                    )
        week_df = pd.DataFrame(rows)
        print(week_df.round(4).to_string(index=False))
        week_df.to_csv('results/weekly_diff.csv', index=False)
        print('=== E1 страты и недели: готово за', round(time.time() - _t0), 'с ===', flush=True)
    except Exception:
        print('!!! ОШИБКА в этапе: E1 страты и недели')
        traceback.print_exc()
else:
    print('[пропуск] E1 страты и недели')


## 6. Избыточность графовых признаков


In [ ]:
graph_names = register_dict({c: TASTE[c].to_numpy() for c in TASTE.columns})
print('граф-признаков зарегистрировано:', len(graph_names))
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform

def redundancy_report(tag):
    """Корреляции Спирмена по подвыборке строк train: кластеры пула v2 и max|rho| новых признаков с пулом."""
    rs_ = np.random.default_rng(0)
    ns_ = min(300000, Xtr_p.shape[0])
    rows_ = rs_.choice(Xtr_p.shape[0], ns_, replace=False)
    names_all = sorted(CP, key=CP.get)
    sub = pd.DataFrame(Xtr_p[rows_][:, [CP[n] for n in names_all]], columns=names_all)
    corr = sub.rank(method='average', na_option='keep').corr(min_periods=1000).abs().fillna(0.0)
    pool_names = list(POOL2_COLS)
    new_names = [n for n in names_all if n not in pool_names]
    Dm = 1 - corr.loc[pool_names, pool_names].to_numpy()
    np.fill_diagonal(Dm, 0)
    Z = linkage(squareform((Dm + Dm.T) / 2, checks=False), method='average')
    out = {thr: int(fcluster(Z, 1 - thr, criterion='distance').max()) for thr in (0.95, 0.9, 0.8)}
    print('кластеры пула v2 (|rho|>порог):', out)
    if new_names:
        red = pd.DataFrame(
            [
                dict(
                    признак=n,
                    max_abs_rho=float(corr.loc[n, pool_names].max()),
                    ближайший=corr.loc[n, pool_names].idxmax(),
                )
                for n in new_names
            ]
        ).sort_values('max_abs_rho', ascending=False)
        print(red.round(3).head(40).to_string(index=False))
        print(
            'доля новых признаков с max|rho|>0,8:',
            round(float((red['max_abs_rho'] > 0.8).mean()), 3),
        )
        red.to_csv(f'results/redundancy_{tag}.csv', index=False)
    pd.Series(out).to_csv(f'results/clusters_{tag}.csv')

def cond_redundancy(names, tag):
    """R^2 регрессии признака на признаках пула v2 (LightGBM; отложенные пользователи)."""
    rs = np.random.default_rng(0)
    users_ = d_tr['uid'].unique()
    hold = set(rs.choice(users_, size=int(0.2 * len(users_)), replace=False).tolist())
    is_hold = d_tr['uid'].isin(hold).to_numpy()
    pool_idx = [CP[c] for c in POOL2_COLS]
    res = []
    for gname in names:
        y = Xtr_p[:, CP[gname]]
        ok = ~np.isnan(y)
        tr_i = np.where(ok & ~is_hold)[0]
        ho_i = np.where(ok & is_hold)[0]
        if len(tr_i) < 1000 or len(ho_i) < 1000:
            continue
        tr_i = rs.choice(tr_i, size=min(300000, len(tr_i)), replace=False)
        ho_i = rs.choice(ho_i, size=min(100000, len(ho_i)), replace=False)
        m = lgb.LGBMRegressor(
            n_estimators=200,
            learning_rate=0.1,
            num_leaves=63,
            subsample=0.8,
            subsample_freq=1,
            colsample_bytree=0.8,
            verbose=-1,
        )
        m.fit(Xtr_p[tr_i][:, pool_idx], y[tr_i])
        pred = m.predict(Xtr_p[ho_i][:, pool_idx])
        r2 = 1 - np.sum((y[ho_i] - pred) ** 2) / np.sum((y[ho_i] - y[ho_i].mean()) ** 2)
        res.append(dict(признак=gname, R2=float(r2)))
        print(gname, round(float(r2), 3), flush=True)
    pd.DataFrame(res).to_csv(f'results/conditional_redundancy_{tag}.csv', index=False)


In [ ]:
if RUN['redundancy']:
    print('=== E2 избыточность (граф) ===', flush=True)
    _t0 = time.time()
    try:
        redundancy_report('graph')
        print(
            '=== E2 избыточность (граф): готово за',
            round(time.time() - _t0), 'с ===', flush=True,
        )
    except Exception:
        print('!!! ОШИБКА в этапе: E2 избыточность (граф)')
        traceback.print_exc()
else:
    print('[пропуск] E2 избыточность (граф)')


In [ ]:
if RUN['cond_redundancy']:
    print('=== E4 условная избыточность (граф) ===', flush=True)
    _t0 = time.time()
    try:
        cond_redundancy(graph_names, 'graph')
        print(
            '=== E4 условная избыточность (граф): готово за',
            round(time.time() - _t0), 'с ===', flush=True,
        )
    except Exception:
        print('!!! ОШИБКА в этапе: E4 условная избыточность (граф)')
        traceback.print_exc()
else:
    print('[пропуск] E4 условная избыточность (граф)')


## 7. Графовые признаки и новые пользовательско-трековые пары


In [ ]:
# Сравниваем графовые признаки с базовым пулом.
if RUN['graph_compare']:
    print('\n' + '=' * 70 + '\n=== ГРАФ ПРОТИВ ПУЛА + НОВЫЕ ПАРЫ ===' + '\n' + '=' * 70, flush=True)
    _t0 = time.time()
    try:
        # Проверяем входные данные.
        graph_cols = list(dict.fromkeys(list(POOL2_COLS) + list(graph_names)))
        print(f'[INFO] базовый пул: {len(POOL2_COLS)} признаков', flush=True)
        print(f'[INFO] графовых признаков: {len(graph_names)}', flush=True)
        print(f'[INFO] общий пул: {len(graph_cols)} признаков', flush=True)
        # Проверяем доступную память перед обучением.
        try:
            import psutil
            process = psutil.Process(os.getpid())
            ram_before = process.memory_info().rss / 1024 ** 3
            print(f'[RAM] перед graph_compare: {ram_before:.2f} GB', flush=True)
        except Exception:
            ram_before = None
        # Отбираем признаки и сравниваем модели.
        print('\n[START] compare_rfe_generic', flush=True)
        sets_graph = compare_rfe_generic(
            graph_cols, 'пул+граф', sets_rfe, Ks=(48, 26), rfe_rounds=220
        )
        print('\n[DONE] compare_rfe_generic', flush=True)
        # Отдельно анализируем новые пары.
        print('\n[START] анализ новых/повторных пар', flush=True)
        mask_new = Xrep_p[:, CP['ui_cnt_400d']] == 0
        print('доля новых пар на rep:', round(float(mask_new.mean()), 3), flush=True)
        rows = []
        for K in (48, 26):
            key_base = f'база_{K}_пул+граф'
            key_new = f'пул+граф_{K}_пул+граф'
            if key_base not in R or key_new not in R:
                print(f'[WARNING] нет результатов для K={K}; пропускаем стратификацию', flush=True)
                continue
            pa = R[key_base]['p']
            pb = R[key_new]['p']
            for nm, mk in [('новые пары', mask_new), ('повторные пары', ~mask_new)]:
                print(f'[STRAT] K={K} | {nm}', flush=True)
                r = strat_diff(pa, pb, mk)
                if r:
                    rows.append(dict(K=K, страта=nm, **r))
                    print(
                        f"K={K}, {nm}: пул+граф - база = "
                        f"{r['d']:+.4f} [{r['lo']:+.4f}; {r['hi']:+.4f}]",
                        flush=True,
                    )
        # Сохраняем таблицы результатов.
        graph_strata = pd.DataFrame(rows)
        graph_strata.to_csv('results/graph_novel_pairs.csv', index=False)
        print('\n[OK] results/graph_novel_pairs.csv сохранён', flush=True)
        # Проверяем использование памяти после этапа.
        gc.collect()
        try:
            process = psutil.Process(os.getpid())
            ram_after = process.memory_info().rss / 1024 ** 3
            print(f'[RAM] после graph_compare: {ram_after:.2f} GB', flush=True)
        except Exception:
            pass
        print(
            '\n' + '=' * 70
            + '\n=== ГРАФ ПРОТИВ ПУЛА + НОВЫЕ ПАРЫ: ГОТОВО ==='
            + f'\nВремя: {round(time.time() - _t0)} с'
            + '\n' + '=' * 70,
            flush=True,
        )
    except Exception:
        print('\n!!! ОШИБКА в этапе: граф против пула + новые пары', flush=True)
        traceback.print_exc()
else:
    print('[пропуск] граф против пула + новые пары', flush=True)


## 8. Явная обратная связь: лайки и дизлайки


In [ ]:
if RUN['feedback']:
    print('=== обратная связь (лайки, дизлайки) ===', flush=True)
    _t0 = time.time()
    try:
        # Строим признаки по лайкам и дизлайкам.
        RAW = find_file('multi_event.parquet', required=False)
        if RAW is None:
            from huggingface_hub import hf_hub_download
            RAW = hf_hub_download(
                repo_id='yandex/yambda',
                repo_type='dataset',
                filename='flat/50m/multi_event.parquet',
                local_dir='data',
            )
        tab = pq.read_table(
            RAW,
            columns=['uid', 'item_id', 'timestamp', 'event_type'],
            filters=[
                ('event_type', 'in', ['like', 'unlike', 'dislike', 'undislike']),
                ('uid', 'in', df['uid'].unique().tolist()),
            ],
        )
        ev = tab.to_pandas()
        del tab
        print(ev['event_type'].value_counts())
        amap, bmap = (
            load_map('artist_item_mapping.parquet'),
            load_map('album_item_mapping.parquet'),
        )
        for col, mp in [('artist_id', amap), ('album_id', bmap)]:
            s = ev['item_id'].map(mp).astype('float64')
            ev[col] = np.where(s.isna(), -1 - ev['item_id'].astype(np.int64), s).astype(np.int64)
        ev['sign'] = ev['event_type'].map({
            'like': 1.0,
            'unlike': -1.0,
            'dislike': 1.0,
            'undislike': -1.0,
        })
        ev['stream'] = ev['event_type'].map({
            'like': 'like',
            'unlike': 'like',
            'dislike': 'dislike',
            'undislike': 'dislike',
        })

        # Добавляем признаки по типам событий и их отношениям.
        def feedback_feats(df, ev_s, keys, windows_days, name, snap=DAY):
            """Чистое число лайков (или дизлайков) за окно до начала суток: (+1 за like, -1 за unlike) по ключу keys."""
            n = len(df)
            allk = pd.concat([df[keys], ev_s[keys]], ignore_index=True)
            codes_all = allk.groupby(keys, sort=False).ngroup().to_numpy(np.int64)
            cq, ce = (codes_all[:n], codes_all[n:])
            te = ev_s['timestamp'].to_numpy(np.int64)
            val = ev_s['sign'].to_numpy(np.float64)
            comb_e = ce << SHIFT | te
            order = np.argsort(comb_e, kind='stable')
            comb_s, val_s = (comb_e[order], val[order])
            cum = np.concatenate([[0.0], np.cumsum(val_s)])
            tq = df['timestamp'].to_numpy(np.int64) // snap * snap
            hi = np.searchsorted(comb_s, cq << SHIFT | tq, side='left')
            out = {}
            for w in windows_days:
                lo = np.searchsorted(comb_s, cq << SHIFT | np.maximum(tq - w * DAY, 0), side='left')
                out[f'{name}_net_{w}d'] = (cum[hi] - cum[lo]).astype(np.float32)
            return out
        # Проверяем результат на небольшой подвыборке.
        _r = np.random.default_rng(3)
        _n = 2500
        _t = pd.DataFrame({
            'uid': _r.integers(0, 8, _n),
            'item_id': _r.integers(0, 12, _n),
            'timestamp': _r.integers(0, 40 * DAY // 5, _n) * 5,
        }).sort_values('timestamp').reset_index(drop=True)
        _e = pd.DataFrame({
            'uid': _r.integers(0, 8, 400),
            'item_id': _r.integers(0, 12, 400),
            'timestamp': _r.integers(0, 40 * DAY // 5, 400) * 5,
            'sign': _r.choice([1.0, -1.0], 400, p=[0.8, 0.2]),
        })
        _F = feedback_feats(_t, _e, ['uid', 'item_id'], [7, 30], 'ui_like')
        for i in _r.choice(_n, 250, replace=False):
            row = _t.iloc[i]
            tq = row.timestamp // DAY * DAY
            for w in (7, 30):
                m = (
                    (_e.uid == row.uid)
                    & (_e.item_id == row.item_id)
                    & (_e.timestamp < tq)
                    & (_e.timestamp >= tq - w * DAY)
                )
                assert abs(_F[f'ui_like_net_{w}d'][i] - _e.sign[m].sum()) < 1e-06
        print('тест feedback_feats пройден')
        FB = {}
        WFB = [30, 90, 400]
        KEYS_FB = {
            'ui': ['uid', 'item_id'],
            'uart': ['uid', 'artist_id'],
            'ualb': ['uid', 'album_id'],
            'user': ['uid'],
        }
        for kname, kcols in KEYS_FB.items():
            for stream in ('like', 'dislike'):
                FB.update(
                    feedback_feats(df, ev[ev['stream'] == stream], kcols, WFB, f'{kname}_{stream}')
                )
        # Считаем отношение лайков к числу прослушиваний.
        for kname in ('user', 'uart', 'ualb'):
            lc = add_window_feats2(df, KEYS_FB[kname], WFB, kname, with_cnt=True)
            for w in WFB:
                FB[f'{kname}_like_per_listen_{w}d'] = (
                    FB[f'{kname}_like_net_{w}d']
                    / (lc[f'{kname}_cnt_{w}d'].to_numpy() + 1.0)
                ).astype(np.float32)
            del lc
        print(len(FB), 'признаков обратной связи:', list(FB)[:6], '...')
        for c in ['ui_like_net_400d', 'uart_like_net_400d', 'user_like_per_listen_400d']:
            v = FB[c][idx_rep]
            print(
                c,
                'ненулевых:', round(float((v != 0).mean()), 4),
                '| NDCG@10 одной фичи:',
                round(
                    ndcg_at_k(
                        d_rep['g'],
                        v + 1e-09 * np.random.default_rng(0).random(len(v)),
                        d_rep['label'],
                    ),
                    4,
                ),
            )
        fb_names = register_dict(FB)
        sets_fb = compare_rfe_generic(list(POOL2_COLS) + fb_names, 'пул+обратная_связь', sets_rfe)
        print('новые признаки в отобранном наборе K=26:', [c for c in sets_fb[26] if c in fb_names])
        print('новые признаки в отобранном наборе K=48:', [c for c in sets_fb[48] if c in fb_names])
        print(
            '=== обратная связь (лайки, дизлайки): готово за',
            round(time.time() - _t0), 'с ===', flush=True,
        )
    except Exception:
        print('!!! ОШИБКА в этапе: обратная связь (лайки, дизлайки)')
        traceback.print_exc()
else:
    print('[пропуск] обратная связь (лайки, дизлайки)')


## 9. Экспоненциальное затухание истории


In [ ]:
if RUN['decay']:
    print('=== ядра затухания ===', flush=True)
    _t0 = time.time()
    try:
        from numba import njit

        @njit(cache=False)
        def _decay_scan(codes_s, ts_s, tq_s, val_s, hl_sec, out_cnt, out_sum):
            n = codes_s.shape[0]
            i = 0
            while i < n:
                c = codes_s[i]
                j = i
                while j < n and codes_s[j] == c:
                    j += 1
                s_cnt = 0.0
                s_sum = 0.0
                t_state = 0.0
                p = i
                for k in range(i, j):
                    tq = tq_s[k]
                    while p < j and ts_s[p] < tq:
                        f = 2.0 ** (-(ts_s[p] - t_state) / hl_sec)
                        s_cnt = s_cnt * f + 1.0
                        s_sum = s_sum * f + val_s[p]
                        t_state = ts_s[p]
                        p += 1
                    f = 2.0 ** (-(tq - t_state) / hl_sec)
                    out_cnt[k] = s_cnt * f
                    out_sum[k] = s_sum * f
                i = j

        def decay_feats(df, keys, half_lives_days, name, target='label', snap=DAY):
            """Экспоненциально взвешенные число и доля по прошлому (вес 2^{-(tau-t_e)/h}), снимок на начало суток."""
            codes = df.groupby(keys, sort=False).ngroup().to_numpy(np.int64)
            ts = df['timestamp'].to_numpy(np.int64)
            val = df[target].to_numpy(np.float64)
            tq = ts // snap * snap
            order = np.argsort(codes << SHIFT | ts, kind='stable')
            codes_s, ts_s, tq_s, val_s = (codes[order], ts[order], tq[order], val[order])
            out = {}
            for h in half_lives_days:
                oc = np.zeros(len(df))
                os_ = np.zeros(len(df))
                _decay_scan(codes_s, ts_s, tq_s, val_s, float(h * DAY), oc, os_)
                rc = np.empty(len(df), np.float32)
                rc[order] = oc
                rr = np.empty(len(df), np.float32)
                rr[order] = np.where(oc > 1e-09, os_ / np.maximum(oc, 1e-09), np.nan)
                out[f'{name}_dcnt_h{h}'] = rc
                out[f'{name}_drate_h{h}'] = rr
            return out
        # Проверяем результат на небольшой подвыборке.
        _r = np.random.default_rng(4)
        _n = 2000
        _t = pd.DataFrame({
            'uid': _r.integers(0, 8, _n),
            'item_id': _r.integers(0, 10, _n),
            'timestamp': _r.integers(0, 40 * DAY // 5, _n) * 5,
            'label': _r.integers(0, 2, _n),
        }).sort_values('timestamp').reset_index(drop=True)
        _F = decay_feats(_t, ['uid', 'item_id'], [3, 14], 'ui')
        for i in _r.choice(_n, 200, replace=False):
            row = _t.iloc[i]
            tq = row.timestamp // DAY * DAY
            prev = _t[(_t.uid == row.uid) & (_t.item_id == row.item_id) & (_t.timestamp < tq)]
            for h in (3, 14):
                wts = 2.0 ** (-(tq - prev.timestamp.to_numpy()) / (h * DAY))
                e_cnt = wts.sum()
                e_rate = (wts * prev.label.to_numpy()).sum() / e_cnt if e_cnt > 1e-09 else np.nan
                assert abs(_F[f'ui_dcnt_h{h}'][i] - e_cnt) < 0.0001 * max(1, e_cnt)
                g = _F[f'ui_drate_h{h}'][i]
                assert np.isnan(e_rate) and np.isnan(g) or abs(e_rate - g) < 0.0001
        print('тест decay_feats пройден')
        HL = [3, 7, 14, 30, 60, 120, 240]
        DF = {}
        for kname, kcols in {
            'ui': ['uid', 'item_id'],
            'uart': ['uid', 'artist_id'],
            'ualb': ['uid', 'album_id'],
            'user': ['uid'],
        }.items():
            DF.update(decay_feats(df, kcols, HL, kname))
            print('ядра:', kname, flush=True)
        # Сравниваем качество отдельных признаков на rep.
        noise_ = 1e-09 * np.random.default_rng(0).random(len(idx_rep))

        def one_feat(v):
            return ndcg_at_k(
                d_rep['g'],
                np.nan_to_num(v, nan=np.nanmean(v)) + noise_,
                d_rep['label'],
            )
        rows = []
        for h in HL:
            rows.append(
                dict(
                    тип='экспоненциальное: доля',
                    параметр=h,
                    ndcg=one_feat(DF[f'ui_drate_h{h}'][idx_rep]),
                )
            )
            rows.append(
                dict(
                    тип='экспоненциальное: число',
                    параметр=h,
                    ndcg=one_feat(DF[f'ui_dcnt_h{h}'][idx_rep]),
                )
            )
        for w in [1, 2, 3, 5, 7, 10, 14, 21, 30, 45, 60, 90, 120, 180, 400]:
            if f'ui_rate_{w}d' in CP:
                rows.append(
                    dict(
                        тип='прямоугольное окно: доля',
                        параметр=w,
                        ndcg=one_feat(Xrep_p[:, CP[f'ui_rate_{w}d']]),
                    )
                )
                rows.append(
                    dict(
                        тип='прямоугольное окно: число',
                        параметр=w,
                        ndcg=one_feat(Xrep_p[:, CP[f'ui_cnt_{w}d']]),
                    )
                )
        curve_df = pd.DataFrame(rows)
        curve_df.to_csv('results/kernel_curves.csv', index=False)
        print(curve_df.pivot(index='параметр', columns='тип', values='ndcg').round(4).to_string())
        # Сравниваем пулы одинакового размера.
        decay_names = register_dict(DF)
        base_item = [c for c in POOL2_COLS if c.startswith(('item_', 'gitem_'))]
        sets_decay_only = compare_rfe_generic(base_item + decay_names, 'только_ядра', sets_rfe)
        sets_decay_all = compare_rfe_generic(list(POOL2_COLS) + decay_names, 'пул+ядра', sets_rfe)
        print('=== ядра затухания: готово за', round(time.time() - _t0), 'с ===', flush=True)
    except Exception:
        print('!!! ОШИБКА в этапе: ядра затухания')
        traceback.print_exc()
else:
    print('[пропуск] ядра затухания')


## 10. E3 — Устойчивость отбора признаков


In [ ]:
if RUN['stability']:
    print('=== E3 устойчивость отбора ===', flush=True)
    _t0 = time.time()
    try:

        def fit_gain_sub(cols, rows_mask, seed=0):
            ci = [CP[c] for c in cols]
            d_ = d_tr.loc[rows_mask]
            sz_ = d_.groupby('g', sort=True).size().to_numpy()
            dtr = lgb.Dataset(
                Xtr_p[rows_mask][:, ci],
                d_['label'].to_numpy(),
                group=sz_,
                params={'max_bin': 63},
            )
            dsel = lgb.Dataset(
                Xsel_p[:, ci], d_sel['label'].to_numpy(), group=sz_sel, reference=dtr
            )
            m = lgb.train(
                {**params_r, 'seed': seed},
                dtr,
                300,
                valid_sets=[dsel],
                callbacks=[
                    lgb.early_stopping(30, verbose=False),
                    lgb.log_evaluation(0),
                ],
            )
            return pd.Series(m.feature_importance('gain'), index=cols)

        def rfe_path_sub(start_cols, targets, rows_mask, drop_frac=0.15):
            cols = list(start_cols)
            out = {}
            for K in sorted(targets, reverse=True):
                while len(cols) > K:
                    imp = fit_gain_sub(cols, rows_mask).sort_values(ascending=False)
                    cols = imp.index[:max(K, int(len(cols) * (1 - drop_frac)))].tolist()
                out[K] = list(cols)
            return out
        users_tr = d_tr['uid'].unique()
        res_sets = {26: [], 48: []}
        for r in range(N_RES):
            ur = np.random.default_rng(100 + r)
            keep_u = set(ur.choice(users_tr, size=int(0.8 * len(users_tr)), replace=False).tolist())
            mk = d_tr['uid'].isin(keep_u).to_numpy()
            t = time.time()
            sets_r = rfe_path_sub(POOL2_COLS, [48, 26], mk)
            for K in (48, 26):
                res_sets[K].append(set(sets_r[K]))
            print(f'подвыборка {r + 1}/{N_RES}: {round(time.time() - t)} с', flush=True)

        def jacc(a, b):
            return len(a & b) / len(a | b)
        stab = []
        for K in (26, 48):
            S = res_sets[K]
            ref = set(sets_rfe[K])
            pair = [jacc(S[i], S[j]) for i in range(len(S)) for j in range(i + 1, len(S))]
            vs_ref = [jacc(s_, ref) for s_ in S]
            freq = pd.Series({
                f: sum((f in s_ for s_ in S)) / len(S)
                for f in POOL2_COLS
            }).sort_values(ascending=False)
            core = int((freq == 1.0).sum())
            stab.append(dict(
                K=K,
                жаккар_между_подвыборками=np.mean(pair),
                жаккар_с_полным_RFE=np.mean(vs_ref),
                ядро_во_всех_подвыборках=core,
            ))
            freq.head(15).to_csv(f'results/selection_frequency_K{K}.csv')
        stab_df = pd.DataFrame(stab)
        print(stab_df.round(3).to_string(index=False))
        stab_df.to_csv('results/selection_stability.csv', index=False)
        print(
            '=== E3 устойчивость отбора: готово за',
            round(time.time() - _t0), 'с ===', flush=True,
        )
    except Exception:
        print('!!! ОШИБКА в этапе: E3 устойчивость отбора')
        traceback.print_exc()
else:
    print('[пропуск] E3 устойчивость отбора')


## 11. AutoFE: random, evolutionary и surrogate


In [ ]:
if RUN['agents']:
    print('=== агенты на 10 seed ===', flush=True)
    _t0 = time.time()
    try:
        multi = {}
        for name, prop in [
            ('random', propose_random),
            ('evo', propose_evo),
            ('surrogate', propose_surrogate),
        ]:
            multi[name] = []
            for s in SEEDS_A:
                print(name, 'seed', s, flush=True)
                multi[name].append(run_agent_cached(name, prop, s))
        rows = []
        for name, runs in multi.items():
            for s, res in zip(SEEDS_A, runs):
                s_rep, p = final_eval(res['accepted'])
                mean, ci = boot_diff_rep(R['M1']['p'], p)
                first = next((i + 1 for i, h in enumerate(res['history']) if h['delta'] > THR),
                    None)
                rows.append(dict(
                    метод=name,
                    seed=s,
                    принято=len(res['accepted']),
                    оценок_до_первого=first,
                    rep=s_rep,
                    d_M1=mean,
                    сек=round(res['sec']),
                ))
                pd.DataFrame(rows).to_csv('results/agents_summary.csv', index=False)
        summary = pd.DataFrame(rows)
        print(summary.round(4).to_string(index=False))
        print(summary.groupby('метод')[['принято', 'rep', 'd_M1']].agg(['mean', 'std']).round(4))
        piv = summary.pivot(index='seed', columns='метод', values='rep')
        paired = pd.DataFrame({
            'evo - random': piv['evo'] - piv['random'],
            'surrogate - random': piv['surrogate'] - piv['random'],
        })
        print(paired.round(4))
        print(paired.mean().round(4))
        paired.to_csv('results/agents_paired.csv')
        # Проверяем избыточность принятых AutoFE-признаков.
        names_acc = sorted({n for runs in multi.values() for res in runs for n in res['accepted']})
        reg_acc = register_dict({n: NEW[n] for n in names_acc if n in NEW})
        print(len(names_acc), 'принятых признаков, зарегистрировано', len(reg_acc))
        redundancy_report('agents')
        cond_redundancy(reg_acc, 'agents')
        sets_v3 = compare_rfe_generic(list(POOL2_COLS) + names_acc, 'пул_v3', sets_rfe)
        print('=== агенты на 10 seed: готово за', round(time.time() - _t0), 'с ===', flush=True)
    except Exception:
        print('!!! ОШИБКА в этапе: агенты на 10 seed')
        traceback.print_exc()
else:
    print('[пропуск] агенты на 10 seed')


## 12. Подготовка последовательных контекстов


In [ ]:
if RUN['seqmodel']:
    print('=== последовательная модель: контексты и проверка утечки ===', flush=True)
    _t0 = time.time()
    try:
        K_CTX = 8 if DEBUG else 64
        _z = np.load(find_file('graph_item_vectors_v1.npz'))
        vec_index_s = pd.Series(np.arange(len(_z['ids'])), index=pd.Index(_z['ids']))
        V_s = _z['V']
        D_VEC = V_s.shape[1]
        uid_all = df['uid'].to_numpy(np.int64)
        ts_all = df['timestamp'].to_numpy(np.int64)
        codes_u = df.groupby('uid', sort=False).ngroup().to_numpy(np.int64)
        uc = pd.Series(codes_u, index=uid_all)
        uc = uc[~uc.index.duplicated()]
        comb_ev = codes_u << SHIFT | ts_all
        ord_ev = np.argsort(comb_ev, kind='stable')
        comb_ev_s = comb_ev[ord_ev]
        item_all = df['item_id'].to_numpy()
        ev_item = item_all[ord_ev]
        ev_y = df['label'].to_numpy()[ord_ev].astype(np.float32)
        ev_pr = df['pr'].to_numpy()[ord_ev].astype(np.float32)
        ev_ts = ts_all[ord_ev]
        _pos = vec_index_s.reindex(ev_item).to_numpy(dtype=np.float64)
        ev_has = ~np.isnan(_pos)
        ev_vpos = np.where(ev_has, _pos, 0).astype(np.int64)
        del _pos

        def build_context(dsub, K=K_CTX):
            """Контекст (пользователь, день): последние K событий пользователя СТРОГО до начала суток + признаки кандидатов."""
            g = dsub['g'].to_numpy()
            ug, inv = np.unique(g,
                return_inverse=True)  # dsub отсортирован по g, поэтому inv не убывает
            uid_g = ug // 1000
            day_start = ug % 1000 * DAY
            ucode = uc.reindex(uid_g).to_numpy().astype(np.int64)
            hi = np.searchsorted(comb_ev_s, ucode << SHIFT | day_start, side='left')
            gstart = np.searchsorted(comb_ev_s, ucode << SHIFT, side='left')
            pos = hi[:, None] - K + np.arange(K)[None, :]
            valid = pos >= gstart[:, None]
            pc = np.where(valid, pos, 0)
            hv = ev_has[pc] & valid
            vec = V_s[ev_vpos[pc]] * hv[..., None]
            dt = np.log1p((day_start[:, None] - ev_ts[pc]) / DAY) / np.log1p(400.0)
            feats = np.concatenate(
                [
                    vec,
                    hv[..., None],
                    (ev_y[pc] * valid)[..., None],
                    (ev_pr[pc] * valid)[..., None],
                    (dt * valid)[..., None],
                ],
                axis=-1,
            ).astype(np.float32)
            rows = dsub.index.to_numpy()
            cv = vec_index_s.reindex(item_all[rows]).to_numpy(dtype=np.float64)
            ch = ~np.isnan(cv)
            cand = np.concatenate(
                [V_s[np.where(ch, cv, 0).astype(np.int64)] * ch[:, None], ch[:, None]],
                axis=1,
            ).astype(np.float32)
            return dict(
                ctx=feats,
                valid=valid,
                cand=cand,
                inv=inv,
                y=dsub['label'].to_numpy().astype(np.float32),
                ug=ug,
                rows=rows,
                day_start=day_start,
                pos=pos,
            )

        def check_context(b, ngroups=100, seed=0):
            r = np.random.default_rng(seed)
            G = len(b['ug'])
            for gi in r.choice(G, min(ngroups, G), replace=False):
                v = b['valid'][gi]
                ts_tok = ev_ts[np.where(v, b['pos'][gi], 0)][v]
                assert (
                    ts_tok < b['day_start'][gi]
                ).all(), 'в контексте есть событие не из прошлого'
                sub = df[
                    (df['uid'] == b['ug'][gi] // 1000)
                    & (df['timestamp'] < b['day_start'][gi])
                ].tail(K_CTX)
                assert len(sub) == int(v.sum())
                assert np.allclose(np.sort(sub['timestamp'].to_numpy().astype(np.int64)),
                    np.sort(ts_tok))
        B_tr, B_sel, B_rep = (build_context(d_tr), build_context(d_sel), build_context(d_rep))
        check_context(B_tr, 60)
        check_context(B_sel, 30)
        print(
            'контексты построены и проверены на отсутствие утечки: '
            'группы train/sel/rep =',
            len(B_tr['ug']), len(B_sel['ug']), len(B_rep['ug']),
            '| форма контекста:', B_tr['ctx'].shape,
        )
        print(
            '=== последовательная модель: контексты и проверка утечки: готово за',
            round(time.time() - _t0), 'с ===', flush=True,
        )
    except Exception:
        print('!!! ОШИБКА в этапе: последовательная модель: контексты и проверка утечки')
        traceback.print_exc()
else:
    print('[пропуск] последовательная модель: контексты и проверка утечки')


## 13. Обучение трансформера


In [ ]:
if RUN['seqmodel']:
    print('=== последовательная модель: обучение ===', flush=True)
    _t0 = time.time()
    try:
        import torch, torch.nn as nn
        dev = 'cuda' if torch.cuda.is_available() else 'cpu'
        EPOCHS = 1 if DEBUG else 5 if dev == 'cuda' else 1
        D_MODEL = 32 if DEBUG or dev == 'cpu' else 64
        MAX_TRAIN_GROUPS = 3000 if DEBUG else None if dev == 'cuda' else 15000
        print('устройство:', dev, '| эпох:', EPOCHS, '| d_model:', D_MODEL)

        class SeqRanker(nn.Module):

            def __init__(self, f_ctx, f_cand, d=64, heads=4, layers=2, K=64):
                super().__init__()
                self.ctx_proj = nn.Linear(f_ctx, d)
                self.pos = nn.Embedding(K + 1, d)
                self.bos = nn.Parameter(torch.zeros(1, 1, d))
                layer = nn.TransformerEncoderLayer(
                    d, heads, dim_feedforward=2 * d, dropout=0.1, batch_first=True
                )
                self.enc = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
                self.cand_proj = nn.Linear(f_cand, d)
                self.cross = nn.MultiheadAttention(d, heads, batch_first=True)
                self.head = nn.Sequential(nn.Linear(2 * d, d), nn.ReLU(), nn.Linear(d, 1))

            def forward(self, ctx, valid, cand, gidx):
                G, K, _ = ctx.shape
                h = self.ctx_proj(ctx)
                h = torch.cat([self.bos.expand(G, 1, -1), h], dim=1) + self.pos(
                    torch.arange(K + 1, device=ctx.device)
                )[None]
                mask = torch.cat(
                    [torch.ones(G, 1, dtype=torch.bool, device=ctx.device), valid], dim=1
                )  # True = валидный токен
                mem = self.enc(h, src_key_padding_mask=~mask)
                q = self.cand_proj(cand)[:, None, :]
                att, _ = self.cross(q, mem[gidx], mem[gidx], key_padding_mask=~mask[gidx])
                return self.head(torch.cat([q[:, 0], att[:, 0]], dim=-1)).squeeze(-1)

        def subset(B, gmask):
            rmask = gmask[B['inv']]
            newidx = np.cumsum(gmask) - 1
            return dict(
                ctx=B['ctx'][gmask],
                valid=B['valid'][gmask],
                cand=B['cand'][rmask],
                inv=newidx[B['inv'][rmask]],
                y=B['y'][rmask],
                rows=B['rows'][rmask],
            )

        def row_ranges(inv, G):
            ar = np.arange(G)
            return (np.searchsorted(inv, ar, 'left'), np.searchsorted(inv, ar, 'right'))

        @torch.no_grad()
        def predict_seq(model, B, max_rows=20000):
            model.eval()
            G = B['ctx'].shape[0]
            rs, re = row_ranges(B['inv'], G)
            out = np.zeros(len(B['y']), np.float32)
            a = 0
            while a < G:
                b = a + 1
                while b < G and re[b] - rs[a] <= max_rows and (b - a < 256):
                    b += 1
                r0, r1 = (rs[a], re[b - 1])
                ctx = torch.from_numpy(B['ctx'][a:b]).to(dev)
                val = torch.from_numpy(B['valid'][a:b]).to(dev)
                cand = torch.from_numpy(B['cand'][r0:r1]).to(dev)
                gidx = torch.from_numpy(B['inv'][r0:r1] - a).to(dev)
                out[r0:r1] = model(ctx, val, cand, gidx).float().cpu().numpy()
                a = b
            return out

        def train_seq(Btr, Bsel, epochs, seed=0, bg=128, max_rows=200, lr=0.002):
            torch.manual_seed(seed)
            rng = np.random.default_rng(seed)
            model = SeqRanker(Btr['ctx'].shape[2], Btr['cand'].shape[1], d=D_MODEL, K=K_CTX).to(dev)
            opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.0001)
            G = Btr['ctx'].shape[0]
            rs, re = row_ranges(Btr['inv'], G)
            best, best_state = (-1.0, None)
            for ep in range(epochs):
                model.train()
                perm = rng.permutation(G)
                if MAX_TRAIN_GROUPS:
                    perm = perm[:MAX_TRAIN_GROUPS]
                tot, nb = (0.0, 0)
                t0 = time.time()
                for a in range(0, len(perm), bg):
                    gb = np.sort(perm[a:a + bg])
                    idx_list, cnts = ([], [])
                    for g_ in gb:
                        r_ = np.arange(rs[g_], re[g_])
                        if len(r_) > max_rows:
                            r_ = rng.choice(r_, max_rows, replace=False)
                        idx_list.append(r_)
                        cnts.append(len(r_))
                    rows_b = np.concatenate(idx_list)
                    gidx = np.repeat(np.arange(len(gb)), cnts)
                    ctx = torch.from_numpy(Btr['ctx'][gb]).to(dev)
                    val = torch.from_numpy(Btr['valid'][gb]).to(dev)
                    cand = torch.from_numpy(Btr['cand'][rows_b]).to(dev)
                    yb = torch.from_numpy(Btr['y'][rows_b]).to(dev)
                    loss = nn.functional.binary_cross_entropy_with_logits(
                        model(ctx, val, cand, torch.from_numpy(gidx).to(dev)), yb
                    )
                    opt.zero_grad()
                    loss.backward()
                    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt.step()
                    tot += float(loss)
                    nb += 1
                s = ndcg_at_k(d_sel['g'], predict_seq(model, Bsel), d_sel['label'])
                print(
                    f'  эпоха {ep + 1}/{epochs}: loss {tot / max(nb, 1):.4f} | '
                    f'sel NDCG@10 {s:.4f} | {round(time.time() - t0)} с',
                    flush=True,
                )
                if s > best:
                    best = s
                    best_state = {k: v.detach().cpu().clone() for k,
                        v in model.state_dict().items()}
            model.load_state_dict(best_state)
            return (model, best)
        # Для train используются предсказания моделей, не обученных на этих пользователях.
        users_tr_ = np.unique(B_tr['ug'] // 1000)
        perm_u = np.random.default_rng(0).permutation(users_tr_)
        fold_of = {u: i % 2 for i, u in enumerate(perm_u)}
        fold_g = np.array([fold_of[u] for u in B_tr['ug'] // 1000])
        seq_all = np.full(len(df), np.nan, np.float32)
        sel_sc, rep_sc = ([], [])
        for f in (0, 1):
            print(f'модель {f + 1}/2: обучение на пользователях другой половины', flush=True)
            model_f, best_sel = train_seq(subset(B_tr, fold_g != f), B_sel, EPOCHS, seed=f)
            Bho = subset(B_tr, fold_g == f)
            seq_all[Bho['rows']] = predict_seq(model_f, Bho)
            sel_sc.append(predict_seq(model_f, B_sel))
            rep_sc.append(predict_seq(model_f, B_rep))
            del model_f
            gc.collect()
        seq_all[B_sel['rows']] = np.mean(sel_sc, 0)
        seq_all[B_rep['rows']] = np.mean(rep_sc, 0)
        np.save('results/seq_scores.npy', seq_all)
        print(
            '=== последовательная модель: обучение: готово за',
            round(time.time() - _t0), 'с ===', flush=True,
        )
    except Exception:
        print('!!! ОШИБКА в этапе: последовательная модель: обучение')
        traceback.print_exc()
else:
    print('[пропуск] последовательная модель: обучение')


## 14. Стекинг: последовательная модель как признак LightGBM


In [ ]:
if RUN['seqmodel']:
    print('=== последовательная модель: стекинг ===', flush=True)
    _t0 = time.time()
    try:
        if 'seq_all' not in globals():
            raise RuntimeError(
                'оценки последовательной модели не получены (этап обучения завершился ошибкой)'
            )
        seq_names = register_dict({'seq_score': seq_all})
        rnd_r = ndcg_at_k(d_rep['g'], np.random.default_rng(0).random(len(d_rep)), d_rep['label'])
        sa = ndcg_at_k(d_rep['g'], seq_all[idx_rep], d_rep['label'])
        print(
            f"последовательная модель отдельно: NDCG@10 rep = {sa:.4f} "
            f"(случайный {rnd_r:.4f}; M1 {R['M1']['rep']:.4f}; "
            f"M2 {R['M2']['rep']:.4f})"
        )
        rows = [dict(набор='seq отдельно', rep=sa)]
        for nm, cols, key in [
            ('M1', FEATS, 'M1'),
            ('RFE_26', sets_rfe[26], 'RFE26_v2'),
            ('RFE_48', sets_rfe[48], 'RFE48_v2'),
            ('POOL2', POOL2_COLS, 'POOL2'),
        ]:
            base_p = get_pred(key, cols)
            _, s_sel, s_rep, p = fit_sel(list(cols) + ['seq_score'])
            mean, ci = boot_diff_rep(base_p, p)
            rows.append(dict(
                набор=nm + ' + seq_score',
                rep=s_rep,
                база=R[key]['rep'],
                d=mean,
                lo=ci[0],
                hi=ci[1],
            ))
            print(
                f"{nm}: со стекингом {s_rep:.4f} против {R[key]['rep']:.4f}; "
                f"Δ = {mean:+.4f} [{ci[0]:+.4f}; {ci[1]:+.4f}]",
                flush=True,
            )
        pd.DataFrame(rows).to_csv('results/seq_stacking.csv', index=False)
        print(
            '=== последовательная модель: стекинг: готово за',
            round(time.time() - _t0), 'с ===', flush=True,
        )
    except Exception:
        print('!!! ОШИБКА в этапе: последовательная модель: стекинг')
        traceback.print_exc()
else:
    print('[пропуск] последовательная модель: стекинг')


## 15. Сохранение результатов


In [ ]:
print('файлы результатов:')
for f in sorted(glob.glob('results/*')):
    print(' ', f, os.path.getsize(f), 'байт')
shutil.make_archive('results_bundle', 'zip', 'results')
print('архив results_bundle.zip создан')


## 16. Выводы по результатам экспериментов

В этом запуске выполнена **часть 2**: автоматический поиск признаков и проверка последовательной модели. Блок `PART = 1` сохранён в ноутбуке, но при текущем `PART = 2` не выполнялся.

### Автоматический поиск признаков

На выборке `rep` средний NDCG@10 составил:

| Стратегия | Средний NDCG@10 | Прирост относительно M1 |
|---|---:|---:|
| `random` | 0.7862 | +0.0070 |
| `evo` | 0.7860 | +0.0068 |
| `surrogate` | 0.7862 | +0.0070 |

Все три стратегии улучшили M1 (0.7792), однако не превзошли POOL2 (0.7894). Заметного преимущества одной стратегии над другими не обнаружено.

Принятые признаки объединили с POOL2, получив пул из 169 признаков, и применили RFE. Результаты после отбора:

| Размер набора | Базовый RFE | RFE из POOL v3 | Разница, 95% ДИ |
|---|---:|---:|---:|
| 48 признаков | 0.7888 | 0.7878 | −0.0010 [−0.0026; +0.0006] |
| 26 признаков | 0.7876 | 0.7864 | −0.0012 [−0.0026; +0.0001] |

Расширение пула новыми кандидатами не улучшило качество после RFE. Оба доверительных интервала включают ноль, поэтому статистически подтверждённого ухудшения также не установлено.

### Последовательная модель

Трансформер получил NDCG@10 = **0.7112** на `rep`: он превосходит случайное ранжирование (0.6950), но уступает LightGBM-моделям. После добавления его предсказания в LightGBM как `seq_score` результаты изменились незначительно. Для POOL2 NDCG@10 вырос с 0.7894 до 0.7904, но 95% доверительный интервал прироста [−0.0004; +0.0024] включает ноль. Для M1 прирост составил +0.0009, также без статистического подтверждения; для RFE-26 эффект был практически нулевым, а для RFE-48 метрика немного снизилась.

### Общий вывод

В проведённых экспериментах AutoFE нашёл признаки, полезные относительно базового M1, но их добавление к уже сильному POOL2 не улучшило отобранные наборы. Последовательная модель извлекла сигнал из порядка взаимодействий, однако её предсказание не обеспечило статистически подтверждённого прироста при стекинге с LightGBM.

**Главный результат:** в текущей конфигурации ни расширение пула за счёт AutoFE, ни добавление последовательного score не продемонстрировали убедительного преимущества над сильным агрегированным baseline. Это вывод именно по проведённым экспериментам, а не утверждение о бесполезности этих методов в принципе.
